# Opus-MT + LoRA: Classical Chinese Poetry → English

Fine-tunes `Helsinki-NLP/opus-mt-zh-en` (MarianMT, ~74M params) on the PoetMT poetry dataset using LoRA.

**Why opus-mt over mT5-base?** opus-mt-zh-en is already pre-trained on OPUS Chinese→English data, so it starts as a working translator. mT5-base was only pre-trained on unsupervised text and cannot translate without extensive fine-tuning first.

**Estimated training time (15 epochs):**
- Free Colab T4: ~20–35 min
- Local GPU (RTX 3060-class): ~50–70 min

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install -q transformers peft accelerate sentencepiece sacrebleu evaluate datasets

In [ ]:
# ── Cell 2: Mount Google Drive (adapter will be saved here) ───────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 3: Clone repo (private) ─────────────────────────────────────────
# Step 1: Colab sidebar → Secrets → add GITHUB_TOKEN → enable Notebook access
# Step 2: Run this cell. If the secret is missing it will prompt you instead.
import os
from getpass import getpass

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = None

if not GITHUB_TOKEN:
    GITHUB_TOKEN = getpass("Paste your GitHub PAT (repo read scope): ")

REPO_DIR = "/content/chinese_poetry_translation"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/emmah-3815/chinese_poetry_translation.git"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git checkout juqy-dev -q
!git pull -q
print("Ready:", os.getcwd())


In [ ]:
# ── Cell 4: Build dataset (if not already built) ──────────────────────────
import os
if not os.path.exists("data/combined/train.jsonl"):
    !python build_dataset.py --poetmt_dir data/PoetMT-main/PoetMT-main/all_poems --ccpm_dir data/CCPM-master --output_dir data/combined
else:
    print("Dataset already built.")

In [ ]:
# ── Cell 5: Smoke test (1 epoch — verify pipeline before full run) ────────
!python pipelines/opus_mt/train_opus_mt.py \n    --data_dir data/combined \n    --output_dir /tmp/opus-mt-smoke \n    --epochs 1 \n    --precision bf16
print("Smoke test done — check output above before running Cell 6")


In [ ]:
# ── Cell 6: Full training (15 epochs, saved to Google Drive) ─────────────
OUTPUT_DIR = "/content/drive/MyDrive/models/opus-mt-poetry"

!python pipelines/opus_mt/train_opus_mt.py \n    --data_dir data/combined \n    --output_dir {OUTPUT_DIR} \n    --epochs 15 \n    --precision bf16


In [ ]:
# ── Cell 7: Evaluate trained adapter on canonical 78-poem test set ───────
OUTPUT_DIR  = "/content/drive/MyDrive/models/opus-mt-poetry"
ADAPTER_DIR = f"{OUTPUT_DIR}/lora_adapter"
EVAL_OUT    = f"{OUTPUT_DIR}/eval_results"

!python eval_e2_mt5.py \n    --adapter_dir {ADAPTER_DIR} \n    --base_model Helsinki-NLP/opus-mt-zh-en \n    --no_task_prefix \n    --flat_test \n    --output_dir {EVAL_OUT}
